# Train SmolVLA on the garage dataset  (Google Colab, T4 GPU)

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Put your dataset on the Hugging Face Hub as a **private dataset** (from your laptop, run `upload_dataset_to_hf.py`). This notebook downloads it straight to Colab's local disk, so nothing has to fit in Google Drive.
3. Create a Hugging Face **read** token (huggingface.co/settings/tokens). Easiest: in Colab click the key icon (Secrets), add a secret named `HF_TOKEN`, and switch on notebook access. Otherwise the download cell asks you to paste it.

**Run the cells top to bottom.** Only run the *Resume* cell if Colab disconnected mid-training.
Checkpoints are trained on Colab's local disk (fast, and Drive can't hold the `last` symlink lerobot makes), and the newest one is copied to Drive every ~90 s. Drive now only holds checkpoints (about 5 GB each), so make sure you have that much free.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Install
If this cell makes Colab ask to restart, do **Runtime -> Restart session** and then continue from the *Settings* cell (do not re-run the install).

In [ ]:
# same lerobot commit the tutorial repo pins (newer lerobot moved lerobot.common and breaks train_model.py)
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git@10b7b3532543b4adfb65760f02a49b4c537afde7"
# huggingface-hub is pinned below 1.0 because transformers 4.50.3 refuses to import with hub 1.x
!pip install -q transformers==4.50.3 num2words accelerate safetensors==0.5.3 datasets==3.4.1 "huggingface-hub==0.36.2"

In [ ]:
import os, shutil, zipfile, subprocess, threading, time
from pathlib import Path

# ----------------------------- SETTINGS -----------------------------
HF_DATASET  = 'YOUR_USERNAME/garage_pnp'                 # <-- the private dataset repo you uploaded from your laptop
DRIVE_DIR   = '/content/drive/MyDrive/garage'            # Drive folder used ONLY for checkpoints
REPO_NAME   = 'garage_pnp'

STEPS       = 15_000       # 8-10k is already a decent first try
BATCH_SIZE  = 16           # 8 if you get CUDA out-of-memory
CHUNK       = 10           # chunk_size = n_action_steps (training and deployment must match)
SAVE_FREQ   = 2_500

WORK        = '/content/work'
DATA_ROOT   = '/content/demo_data_garage'                # dataset is downloaded to local disk (fast for images)
LOCAL_OUT   = '/content/ckpt/smolvla_garage'
DRIVE_OUT   = f'{DRIVE_DIR}/ckpt/smolvla_garage'
# --------------------------------------------------------------------
os.makedirs(WORK, exist_ok=True)
assert 'YOUR_USERNAME' not in HF_DATASET, "set HF_DATASET to your Hugging Face dataset repo, e.g. 'myname/garage_pnp'"

In [ ]:
# dataset: download from the Hugging Face Hub straight to Colab's local disk (resumable if it is interrupted)
from huggingface_hub import login, snapshot_download

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
login(token=token) if token else login()

if not os.path.exists(f'{DATA_ROOT}/meta/info.json'):
    snapshot_download(repo_id=HF_DATASET, repo_type='dataset', local_dir=DATA_ROOT, max_workers=8)
assert os.path.exists(f'{DATA_ROOT}/meta/info.json'), "download finished but meta/info.json is missing - check the repo layout (data/ and meta/ at the top level)"

# Training builds a second copy of the data (a cache) next to the files, so check the free space.
!df -h /content

# training script (identical to the one in the tutorial repo)
!wget -q -O {WORK}/train_model.py https://raw.githubusercontent.com/jeongeun980906/lerobot-mujoco-tutorial/master/train_model.py
print(os.listdir(WORK))

In [ ]:
# sanity-check the dataset before spending GPU time
import json
info = json.load(open(f'{DATA_ROOT}/meta/info.json'))
print("episodes:", info['total_episodes'], " frames:", info['total_frames'], " fps:", info['fps'])
for k, v in info['features'].items():
    print(f"  {k:28s} {v['dtype']:8s} {v['shape']}")
print("tasks:")
for line in open(f'{DATA_ROOT}/meta/tasks.jsonl'):
    print("  ", json.loads(line))
assert info['features']['observation.state']['shape'] == [8] and info['features']['action']['shape'] == [8]

In [ ]:
yaml_text = f'''
dataset:
  repo_id: {REPO_NAME}
  root: {DATA_ROOT}
policy:
  type: smolvla
  chunk_size: {CHUNK}
  n_action_steps: {CHUNK}
  device: cuda
  use_amp: false            # T4 has no bf16
output_dir: {LOCAL_OUT}
batch_size: {BATCH_SIZE}
job_name: smolvla_garage
resume: false
seed: 42
num_workers: 2
steps: {STEPS}
eval_freq: -1
log_freq: 50
save_checkpoint: true
save_freq: {SAVE_FREQ}
use_policy_training_preset: true
wandb:
  enable: false
'''
YAML = f'{WORK}/smolvla_garage.yaml'
open(YAML, 'w').write(yaml_text)
print(yaml_text)

In [ ]:
def latest_local_ckpt():
    last = Path(LOCAL_OUT) / 'checkpoints' / 'last'
    return Path(os.path.realpath(last)) if last.exists() else None     # 'last' is only updated AFTER a save finished

def sync_to_drive():
    src = latest_local_ckpt()
    if src is None:
        return
    root = Path(DRIVE_OUT) / 'checkpoints'
    dst = root / src.name
    if dst.exists():
        return
    root.mkdir(parents=True, exist_ok=True)
    tmp = root / (src.name + '.tmp')
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp, symlinks=False)
    os.rename(tmp, dst)
    for old in root.glob('[0-9]*'):                 # keep only the newest on Drive (each one is several GB)
        if old.name != src.name and not old.name.endswith('.tmp'):
            shutil.rmtree(old, ignore_errors=True)
    print(f"[sync] checkpoint {src.name} saved to Drive")

### Train  (a few hours on a T4 for 15k steps; output is streamed below)

In [ ]:
stop = threading.Event()
def syncer():
    while not stop.is_set():
        try: sync_to_drive()
        except Exception as e: print("[sync] error:", e)
        stop.wait(90)

th = threading.Thread(target=syncer, daemon=True); th.start()
p = subprocess.Popen(f"python train_model.py --config_path {YAML}", shell=True, cwd=WORK,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
p.wait()
stop.set(); th.join()
sync_to_drive()
print("training process exited with code", p.returncode)

### [Only if Colab disconnected] Resume
Re-run the cells **Mount Drive, Settings, dataset download, yaml, sync helpers** (not the install if the runtime was kept), then run this cell.

In [ ]:
def restore_from_drive():
    root = Path(DRIVE_OUT) / 'checkpoints'
    cks = sorted(p for p in root.glob('[0-9]*') if not p.name.endswith('.tmp'))
    assert cks, "no checkpoint on Drive yet"
    src = cks[-1]
    dst_root = Path(LOCAL_OUT) / 'checkpoints'
    dst_root.mkdir(parents=True, exist_ok=True)
    if not (dst_root / src.name).exists():
        shutil.copytree(src, dst_root / src.name)
    last = dst_root / 'last'
    if last.is_symlink() or last.exists():
        last.unlink()
    os.symlink(src.name, last)
    return dst_root / 'last' / 'pretrained_model' / 'train_config.json'

cfg_json = restore_from_drive()
stop = threading.Event()
th = threading.Thread(target=syncer, daemon=True); th.start()
p = subprocess.Popen(f"python train_model.py --config_path={cfg_json} --resume=true", shell=True, cwd=WORK,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
p.wait(); stop.set(); th.join(); sync_to_drive()

### Finished: where is my model?

In [ ]:
root = Path(DRIVE_OUT) / 'checkpoints'
for c in sorted(root.glob('[0-9]*')):
    print(c, '->', [x.name for x in (c / 'pretrained_model').iterdir()])
print()
print("Download the folder  <that path>/pretrained_model  to your laptop, e.g. into")
print("  ./ckpt/smolvla_garage/checkpoints/<step>/pretrained_model")